# Olivine-to-wadsleyite transition rate laws

This notebook sets up the phase boundary, growth law, and nucleation law for a composite olivine-to-wadsleyite transition kinetics. It then evaluates rates at fixed pressure and temperature.

## 1. Set the model parameters

The transition boundary contains pressure in pascals, temperature in kelvin, and a Clapeyron slope in pascals per kelvin. 

The growth parameters are for the interface-controlled law from Hosoya, 2005 (TODO: reference) 

The nucleation parameters are for the surface law from (TODO: reference)

The transition parameters set the driving force and convert surface nucleation to a rate per unit volume.

In [ ]:
from math import exp

import pandas as pd

from gdmate.rheology_models.transition_kinetics import (
    InterfaceControlledGrowth,
    InterfaceControlledGrowthParameters,
    SurfaceNucleation,
    SurfaceNucleationParameters,
    TransitionBoundary,
    TransitionBoundaryParameters,
    TransitionKinetics,
    TransitionKineticsParameters,
)

boundary_parameters = TransitionBoundaryParameters(
    reference_pressure=13.5e9,
    reference_temperature=1740.0,
    clapeyron_slope=2e6,
)
growth_parameters = InterfaceControlledGrowthParameters(
    prefactor=exp(-18),
    water_exponent=3.2,
    activation_enthalpy=274e3,
    activation_volume=3.3e-6,
)
nucleation_parameters = SurfaceNucleationParameters(
    prefactor=1e30,
    activation_enthalpy=274e3,
    activation_volume=3.3e-6,
    interfacial_energy=0.46,
    shape_factor=6e-4,
    molar_volume=4.05e-5,
)
transition_parameters = TransitionKineticsParameters(
    molar_volume_change=3.16e-6,
    grain_boundary_area_factor=6.7,
)

## 2. Put the model together

Pass the transition-boundary object and both laws to `TransitionKinetics`. 

In [ ]:
boundary = TransitionBoundary(boundary_parameters)
kinetics = TransitionKinetics(
    boundary=boundary,
    growth_law=InterfaceControlledGrowth(growth_parameters),
    nucleation_law=SurfaceNucleation(nucleation_parameters),
    parameters=transition_parameters,
)

print("Growth process:", kinetics.growth_law.process_type)
print("Nucleation process:", kinetics.nucleation_law.process_type)

## 3. Check the rates at fixed conditions

The first three rows have pressure above the phase boundary. The last row is on the boundary, where the forward growth and nucleation rates are zero. 

Water content is in weight ppm and initial grain size is in metres.

In [ ]:
pressure = 13.5e9
temperature = [873.15, 973.15, 1573.15, 1740.0]
rates = kinetics.rates(
    pressure,
    temperature,
    water_content_ppm=150.0,
    initial_grain_size=1e-2,
)

pd.DataFrame({
    "Temperature (K)": temperature,
    "Pressure (GPa)": pressure / 1e9,
    "Equilibrium pressure (GPa)": boundary.pressure_at(temperature) / 1e9,
    "Growth (m/s)": rates.growth,
    "Surface nucleation (1/m²/s)": rates.surface_nucleation,
    "Volumetric nucleation (1/m³/s)": rates.volumetric_nucleation,
})